In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os
# import importlib
# import utility.deduplication
# importlib.reload(utility.deduplication)
from datetime               import datetime, timezone
from pyspark.sql            import functions as F
sys.path.insert(0, "/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/")

from config.config_load     import load_config
from utils.logger           import get_logger, log_job
# from utils.deduplication    import deduplicate
# from utils.silver_write     import merge_silver
from utils.dq.def_dq_rules  import get_full_table_name, dq_check_gender_vs_nik, apply_rule_check, run_dq_rule

In [0]:
# JOB CONFIGURATION -----------
JOB_NAME   = "gold_dq_profiling"

DQ_CONFIG_PATH = f"/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/config/dq_rules.yaml"
# LOGGER -----------

logger = get_logger(JOB_NAME)
logger.info("Table Configuration loaded successfully")

# LOAD JOB AND TABLE CONFIGURATION -------------
logger.info("Loading configuration")
cfg = load_config(DQ_CONFIG_PATH)

table = cfg["tables"][JOB_NAME]

catalog       = cfg["catalog"]
schemas        = cfg["schemas"]
# target_schema = cfg["schemas"]["silver"]

silver_table        = f"{catalog}.{schemas}.{table['silver_table']}"
gold_table_summary  = f"{catalog}.{schemas}.{table['gold_table_summary']}"
gold_table_summary  = f"{catalog}.{schemas}.{table['gold_table_detail']}"
log_table           = f"{catalog}.{schemas}.log_jobs"


logger.info("Schema and Table Configuration loaded successfully")


In [0]:
def run_dq_pipeline(spark, config: dict, run_id: str):
    catalog = config["catalog"]
    schema  = config["schemas"]
    rules   = config.get("dq_rules", [])

    all_summary = []
    all_detail  = []

    for rule in rules:
        full_table  = get_full_table_name(catalog, schema, rule["table_name"])
        primary_key = rule["primary_key"]
        df = spark.read.table(full_table)

        df_checked, check_column = apply_rule_check(df, rule)

        summary, detail = run_dq_rule(
            df=df_checked,
            rule_name=rule["rule_name"],
            rule_category=rule["rule_category"],
            column_name=rule["column_name"],
            severity=rule["severity"],
            check_column=check_column,
            # rule_description=rule["rule_description"],
            table_name=full_table,
            run_id=run_id,
            primary_key=primary_key,
            checked_column=rule.get("checked_column")
        )

        all_summary.append(summary)
        all_detail.append(detail)

    final_summary = all_summary[0]
    for s in all_summary[1:]:
        final_summary = final_summary.unionByName(s)

    final_detail = all_detail[0]
    for d in all_detail[1:]:
        final_detail = final_detail.unionByName(d, AllowMissingColumns=True)

    return final_summary, final_detail

In [0]:
RUN_ID_TEST = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
summary_df, detail_df = run_dq_pipeline(spark, cfg, RUN_ID_TEST)

detail_df.display()